In [ ]:
# allows update of external libraries without need to reload package
%load_ext autoreload
%autoreload 2

# Ruggedness index and some demonstrations

## Summary

Demonstrate and visualize the computation of the ruggedness index RIX on various maps.

## Results

### Key results

- For each example map, compute:
  - Compute steep parts as defined in the ruggedness index.
  - Show these steep parts on the map.
  - Show the (not-yet-aggregated) angular RIX values.
- Examples are sunny-weather examples: A planar map, an egg box map and a radial wave map.
- Critical parts:
  - Interpolation method of 2D maps to 1D rays. Artifacts introduced need to be identified.
  - Floatiness of the slopes and the critical slope: The binary comparison is unstable if the slope along a ray is often close to the critical slope (check out the radial wave map).

### Details

1. Requirements:
   - A map given as a `xarray.DataArray` given in its own coordinate system with coordinates `("y", "x")`.
   - A location on the map encoded in a `LocationCCS` object (given as easting and northing).
1. Parameters:
   - `R_km`: Radius [km] of the disc of evaluation. TR6: 3.5km.
   - `dr_km`: Stepsize between points on each ray. `R_km` shall be a multiple of `dr_km`.
1. Nomenclature:
   - `RayGeometry` is the ray starting from a given location in a given direction of length `R_km`.
   - `FieldSampler` samples from the map at given points.
     - Typically, these points are specified by `RayGeometry`.
   - `RayProfile` evaluates the map profile, and determines rix and steep segments.
   - Any slope used here is measured in [m/m] as in TR6, especially not [°].
1. Examples

## Remarks

1. Plots:
   - `plot_profile_and_steep_segments`: The slopes and its mask of steep slopes represent left-labelled intervals, and are plotted for simplicity at their label, i.e. at the left bound. The artifact is of visual nature.

## Imports and Setup

In [ ]:
import numpy as np
import pyproj
import scipy
import xarray
import pandas as pd
import geopandas as gpd
import shapely.geometry
import matplotlib.colors
import matplotlib.pyplot as plt

import ergaleiothiki
import phoibe.artificial_data.fields
from phoibe.geography.complexity.rix import RayGeometry, RayProfile, compute_radial_rix, NaNPolicy
from phoibe.geography.complexity.sampler import RegularGridXYSampler
from iris_datasets_get import get_elevation_data

In [ ]:
LAND_CMAP = matplotlib.colors.LinearSegmentedColormap.from_list(
    "land_cmap", [(0.00, "#2e7d32"), (0.30, "#66bb6a"), (0.55, "#a1887f"), (0.75, "#9e9e9e"), (1.00, "#ffffff")]
)

## Helper functions

### Plotting

In [ ]:
def plot_geodata(da: xarray.DataArray, cmap=LAND_CMAP):
    """Plot the elevation map in colours."""
    figure = plt.figure()
    ax = figure.add_subplot(1, 1, 1)
    cnorm = matplotlib.colors.Normalize(vmin=da.min(), vmax=da.max())
    mesh = ax.pcolormesh(da["x"], da["y"], da, cmap=cmap, shading="auto", norm=cnorm)

    ax.grid(visible=True)

    cbar = plt.colorbar(mesh, ax=ax, orientation="vertical", shrink=0.7)
    cbar.set_label("Elevation [m]")
    plt.tight_layout()
    return figure, ax


def plot_profile_and_steep_segments(ray_profile, slope_critical):
    """Plot the elevation profile and its slopes along a ray, and marks steep parts."""
    r_m = ray_profile.r_m[:-1]
    z = ray_profile.z[:-1]
    dz = ray_profile.slopes
    steep_mask = ray_profile.steep_mask(slope_critical=slope_critical)

    figure, axes = plt.subplots(1, 2, squeeze=True, figsize=(17, 7))
    axes[0].plot(r_m, z)
    axes[0].plot(r_m[steep_mask], z[steep_mask], c="r")
    axes[0].set_title(f"Profile along ray for {slope_critical=}")

    axes[1].plot(r_m, dz)
    axes[1].hlines(y=slope_critical, xmin=r_m[0], xmax=r_m[-1], color="r")
    axes[1].hlines(y=-slope_critical, xmin=r_m[0], xmax=r_m[-1], color="r")
    axes[1].set_title(f"Slope along ray for {slope_critical=}")

    return figure, axes

### Computation and evaluation of intermediate and final results

In [ ]:
def evaluate_rix_for_ray(ray_profile: RayProfile, slope_critical: float):
    """Evaluate RIX and related numbers."""
    n_steep_segments = ray_profile.steep_mask(slope_critical).sum()
    n_segments = ray_profile.slopes.size
    rix = ray_profile.rix(slope_critical)
    message = (
        f"Number of steep segments {n_steep_segments} vs total number of segments {n_segments} vs. RIX {rix:2.4f}."
    )
    return message


def compute_radial_rix(location, sampler, n_angles, R_km, dr_km, slope_critical):
    """For radial rays, determine their individual steepness fractions and steep segments."""
    angles = np.linspace(0, 360, n_angles, endpoint=False)
    ray_profiles = []
    rix_values = []
    lines = []
    steep_ray_segments = []

    for theta in angles:
        ray = RayGeometry(location=location, theta=theta, R_km=R_km, dr_km=dr_km)
        ray_profile = RayProfile(ray=ray, sampler=sampler, nan_policy=NaNPolicy.ERROR)
        ray_profiles.append(ray_profile)
        rix_values.append(ray_profile.rix(slope_critical=slope_critical))
        line = shapely.geometry.LineString([(ray.xs[k], ray.ys[k]) for k in [0, -1]])
        lines.append(gpd.GeoDataFrame({"theta": [theta]}, geometry=[line]))
        segments = ray_profile.steep_ray_segments(slope_critical=slope_critical)
        steep_ray_segments.append(gpd.GeoDataFrame({"theta": [theta] * len(segments)}, geometry=segments))

    steep_ray_segments = pd.concat(steep_ray_segments)
    lines = pd.concat(lines)

    result = {
        "theta": angles,
        "rix_values": rix_values,
        "ray_profiles": ray_profiles,
        "steep_ray_segments": steep_ray_segments,
        "lines": lines,
    }
    return result

## A planar map

In [ ]:
NX, NY = 201, 201
DX, DY = 10, 10
SLOPE_X, SLOPE_Y = 0.5, 0

LOCATION = ergaleiothiki.perdix.LocationCCS(easting=0, northing=0, zone=32)
R_km = 1.5
DR_km = 0.01
THETA = 90

plane = phoibe.artificial_data.fields.make_planar_field(nx=NX, ny=NY, dx=DX, dy=DY, slope_x=SLOPE_X, slope_y=SLOPE_Y)
figure, ax = plot_geodata(da=plane)
ax.scatter(x=LOCATION.easting, y=LOCATION.northing, c="r")

ray = RayGeometry(location=LOCATION, theta=THETA, R_km=R_km, dr_km=DR_km)
elevation_sampler = RegularGridXYSampler(da=plane, method="linear")
ray_profile = RayProfile(ray=ray, sampler=elevation_sampler, nan_policy=NaNPolicy.ERROR)


slope_critical = 0.49
_ = plot_profile_and_steep_segments(ray_profile, slope_critical)
evaluate_rix_for_ray(ray_profile, slope_critical)

slope_critical = 0.5
_ = plot_profile_and_steep_segments(ray_profile, slope_critical)
evaluate_rix_for_ray(ray_profile, slope_critical)

In [ ]:
N_ANGLES = 72
SLOPE_CRITICAL = 0.25

result = compute_radial_rix(
    LOCATION, elevation_sampler, n_angles=N_ANGLES, R_km=R_km, dr_km=DR_km, slope_critical=SLOPE_CRITICAL
)

figure, ax = plot_geodata(da=plane)
ax.scatter(x=LOCATION.easting, y=LOCATION.northing, c="r")
ax.set_title(f"Map with rays and their steep segments, RIX={np.mean(result['rix_values']):.2f}")

result["lines"].plot(ax=ax, color="b", linewidth=0.2, label="ray")
result["steep_ray_segments"].plot(ax=ax, color="r", linewidth=1, label="ray's steep part")
ax.legend()

figure = plt.figure()
ax = figure.add_subplot(1, 1, 1)
pd.Series(data=result["rix_values"], index=result["theta"]).plot(ax=ax);

## Same planar map, different location

In [ ]:
LOCATION = ergaleiothiki.perdix.LocationCCS(easting=997, northing=965, zone=32)
R_km = 1
DR_km = 0.01
THETA = 90

ray = RayGeometry(location=LOCATION, theta=THETA, R_km=R_km, dr_km=DR_km)
ray.r_m.shape
elevation_sampler = RegularGridXYSampler(da=plane, method="nearest")
ray_profile = RayProfile(ray=ray, sampler=elevation_sampler, nan_policy=NaNPolicy.ERROR)
pd.DataFrame({"r": ray_profile.r_m, "z": ray_profile.z}).plot(x="r", y="z")

In [ ]:
kwargs_rix = {"n_angles": 72, "R_km": R_km, "dr_km": DR_km}
compute_radial_rix(location_ccs=LOCATION, sampler=elevation_sampler, slope_critical=1, **kwargs_rix)[0]

## An egg box map

In [ ]:
NX, NY = 201, 201
DX, DY = 10, 10
FREQ_X, FREQ_Y = 0.010, 0.010

LOCATION = ergaleiothiki.perdix.LocationCCS(easting=0, northing=0, zone=32)
R_km = 2
DR_km = 0.01
THETA = 90

eggbox = phoibe.artificial_data.fields.make_eggbox_field(nx=NX, ny=NY, dx=DX, dy=DY, freq_x=FREQ_X, freq_y=FREQ_Y)
figure, ax = plot_geodata(da=eggbox)
ax.scatter(x=LOCATION.easting, y=LOCATION.northing, c="r")

ray = RayGeometry(location=LOCATION, theta=THETA, R_km=R_km, dr_km=DR_km)
elevation_sampler = RegularGridXYSampler(da=eggbox, method="linear")
ray_profile = RayProfile(ray=ray, sampler=elevation_sampler, nan_policy=NaNPolicy.ERROR)


slope_critical = 0.0075
_ = plot_profile_and_steep_segments(ray_profile, slope_critical)
evaluate_rix_for_ray(ray_profile, slope_critical)

slope_critical = 0.009
_ = plot_profile_and_steep_segments(ray_profile, slope_critical)
evaluate_rix_for_ray(ray_profile, slope_critical)

In [ ]:
N_ANGLES = 72
SLOPE_CRITICAL = 0.0025

result = compute_radial_rix(
    LOCATION, elevation_sampler, n_angles=N_ANGLES, R_km=R_km, dr_km=DR_km, slope_critical=SLOPE_CRITICAL
)

figure, ax = plot_geodata(da=eggbox)
ax.set_title(f"Map with rays and their steep segments, RIX={np.mean(result['rix_values']):.2f}")

result["lines"].plot(ax=ax, color="b", linewidth=0.2, label="ray")
result["steep_ray_segments"].plot(ax=ax, color="r", linewidth=1, label="ray's steep part")
ax.scatter(x=LOCATION.easting, y=LOCATION.northing, c="black")
ax.legend()

figure = plt.figure()
ax = figure.add_subplot(1, 1, 1)
pd.Series(data=result["rix_values"], index=result["theta"]).plot(ax=ax);

In [ ]:
LOCATION = ergaleiothiki.perdix.LocationCCS(easting=75, northing=-215, zone=32)
R_km = 1

N_ANGLES = 72
SLOPE_CRITICAL = 0.0025

result = compute_radial_rix(
    LOCATION, elevation_sampler, n_angles=N_ANGLES, R_km=R_km, dr_km=DR_km, slope_critical=SLOPE_CRITICAL
)

figure, ax = plot_geodata(da=eggbox)
ax.set_title(f"Map with rays and their steep segments, RIX={np.mean(result['rix_values']):.2f}")

result["lines"].plot(ax=ax, color="b", linewidth=0.2, label="ray")
result["steep_ray_segments"].plot(ax=ax, color="r", linewidth=1, label="ray's steep part")
ax.legend()

ax.scatter(x=LOCATION.easting, y=LOCATION.northing, c="black")

figure = plt.figure()
ax = figure.add_subplot(1, 1, 1)
pd.Series(data=result["rix_values"], index=result["theta"]).plot(ax=ax);

## A radial wave map


In [ ]:
NX, NY = 201, 201
DX, DY = 10, 10
FREQ = 4

LOCATION = ergaleiothiki.perdix.LocationCCS(easting=0, northing=0, zone=32)
R_km = 2
DR_km = 0.01
THETA = 90

radial_wave = phoibe.artificial_data.fields.make_radial_wave_field(nx=NX, ny=NY, dx=DX, dy=DY, freq=FREQ)
figure, ax = plot_geodata(da=radial_wave)
ax.scatter(x=LOCATION.easting, y=LOCATION.northing, c="r")

ray = RayGeometry(location=LOCATION, theta=THETA, R_km=R_km, dr_km=DR_km)
elevation_sampler = RegularGridXYSampler(da=radial_wave, method="linear")
ray_profile = RayProfile(ray=ray, sampler=elevation_sampler, nan_policy=NaNPolicy.ERROR)


slope_critical = 0.0075
_ = plot_profile_and_steep_segments(ray_profile, slope_critical)
evaluate_rix_for_ray(ray_profile, slope_critical)

In [ ]:
N_ANGLES = 72
SLOPE_CRITICAL = 1e-3

result = compute_radial_rix(
    LOCATION, elevation_sampler, n_angles=N_ANGLES, R_km=R_km, dr_km=DR_km, slope_critical=SLOPE_CRITICAL
)

figure, ax = plot_geodata(da=radial_wave)
ax.set_title(f"Map with rays and their steep segments, RIX={np.mean(result['rix_values']):.2f}")

result["lines"].plot(ax=ax, color="b", linewidth=0.2, label="ray")
result["steep_ray_segments"].plot(ax=ax, color="r", linewidth=1, label="ray's steep part")
ax.legend()

_ = ax.scatter(x=LOCATION.easting, y=LOCATION.northing, c="black")

figure = plt.figure()
ax = figure.add_subplot(1, 1, 1)
pd.Series(data=result["rix_values"], index=result["theta"]).plot(ax=ax)